In [95]:
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
import sys, os
from ipywidgets import interact
from IPython.display import display, Math
from matplotlib.gridspec import GridSpec
from matplotlib.colors import LinearSegmentedColormap
from IPython.display import display, Math

## Functions

In [ ]:
me = 9.109e-31 #kg
h = 6.626e-34 #J s
hbar = h/(np.pi*2)



cmap = LinearSegmentedColormap.from_list(
    "bright_div",
    [ "black","red"])
cmap1 = LinearSegmentedColormap.from_list(
    "bright_div",
    [ "black","blue"])

cmap2 = LinearSegmentedColormap.from_list(
    "bright_div",
    [ "red","black","blue"])

In [155]:
def free_part():
    cmap2 = LinearSegmentedColormap.from_list(
    "bright_div",
    [ "purple","tab:blue","tab:orange"])
    npnts=2000
    wl = h*1e24
    x = np.linspace(0, 2*wl, npnts)
    @interact(
    p = widgets.FloatSlider(min=-3, max=3, value=.5, description='p × 10²⁴='),
    elev= widgets.FloatSlider(min=0, max=90, value=30, description='Elevation'),
    azim= widgets.FloatSlider(min=0, max=90, value=60, description='Rotate'),
    pos = widgets.FloatSlider(min=0, max=2*wl*1e9,step=.0005, description='Point (nm)'),
    Real = widgets.ToggleButton(),
    Imaginary = widgets.ToggleButton()
    )
    def g(p, Real, Imaginary, elev, azim, pos):
        fig = plt.figure(figsize=(8,4))
        gs = GridSpec(1,2,fig, wspace=0.0)
        ax = []
        ax.append(fig.add_subplot(gs[1], projection='3d'))
        ax.append(fig.add_subplot(gs[0]))

        p*=1e-24
        wl = h*1e24
        x = np.linspace(0, 2*wl, npnts)
        psi = np.exp(1j*p/hbar*x)
        y = np.real(psi)
        z = np.imag(psi)

        x_comb = x
        y_comb = y
        z_comb = z
        c = np.ones_like(x)*-1

        if Real:
            x_comb = np.concatenate([x_comb, x])
            y_comb = np.concatenate([y_comb, y])
            z_comb = np.concatenate([z_comb, np.zeros_like(z)])
            c = np.concatenate([c, np.zeros_like(x)])
                    
        if Imaginary:
            x_comb = np.concatenate([x_comb,x])
            y_comb = np.concatenate([y_comb,  np.zeros_like(y)])
            z_comb = np.concatenate([z_comb, z])
            c = np.concatenate([c, np.ones_like(x)])    

        pos = np.abs(int(npnts *pos/(2*wl*1e9))-1)
        x_pos = np.array([x[pos],x[pos]])
        y_pos = np.array([0., y[pos]])
        z_pos = np.array([0., z[pos]])

        ax[0].plot(x*1e9, np.zeros_like(x), np.zeros_like(x), c='k')
        ax[0].plot(x_pos*1e9, y_pos, z_pos, c='red')
        ax[0].scatter(x_comb*1e9, y_comb, z_comb, s=.1, alpha=1, c=c, vmin=-1, vmax=1,cmap=cmap2)
        ax[0].scatter(x[pos]*1e9,0,0, c='k')
        if Real:
            ax[0].scatter(x[pos]*1e9,y[pos],0, c='tab:blue')
        if Imaginary:
            ax[0].scatter(x[pos]*1e9,0,z[pos], c='tab:orange')
        

        ax[0].view_init(elev=elev, azim=-azim)
        ax[0].set_ylim(-1,1)
        ax[0].set_zlim(-1,1)
        ax[0].set_xlabel('x (nm)')
        ax[0].set_ylabel(f'Real($\\psi$)')
        ax[0].set_zlabel(f'Imag($\\psi$)',labelpad=-.5)

        ax[1].plot(x*1e9, np.zeros_like(x), c='k')
        ax[1].plot(x*1e9,y, label=f'Real($\\psi$)')
        ax[1].scatter(x[pos]*1e9,y[pos],c='tab:blue')
        ax[1].plot(x*1e9,z, label=f'Imag($\\psi$)')
        ax[1].scatter(x[pos]*1e9,z[pos],c='tab:orange')
        ax[1].set_xlabel('x (nm)')
        ax[1].set_ylabel(f'$\\psi$')
        ax[1].set_ylim(-1.1,1.1)
        ax[1].legend()

        #display(Math(rf"\psi = e^{{i p x/\hbar}}= e^{{i 2\pi x/({wl:.2f}nm)}}"))
        ax[1].set_title(f"$\\psi = e^{{i p x/\\hbar}}$")
        plt.show()

In [191]:
def super_free_part():
    cmap2 = LinearSegmentedColormap.from_list(
    "bright_div",
    [ "purple","tab:blue","tab:orange"])
    npnts=2000
    wl = h*1e24
    x = np.linspace(0, 2*wl, npnts)
    @interact(
    p = widgets.FloatSlider(min=-3, max=3, value=.5, description='p1 × 10²⁴='),
    p2 = widgets.FloatSlider(min=-3, max=3, value=-.5, description='p2 × 10²⁴='),
    c1s = widgets.FloatSlider(min=0, max=1,  value=1, step=.01, description='c1^2'),
    elev= widgets.FloatSlider(min=0, max=90, value=30, description='Elevation'),
    azim= widgets.FloatSlider(min=0, max=90, value=60, description='Rotate'),
    pos = widgets.FloatSlider(min=0, max=2*wl*1e9,step=.0005, description='Point (nm)'),
    Real = widgets.ToggleButton(),
    Imaginary = widgets.ToggleButton()
    )
    def g(p, p2,c1s, Real, Imaginary, elev, azim, pos):
        c1 = c1s**.5
        fig = plt.figure(figsize=(8,4))
        gs = GridSpec(1,2,fig, wspace=0.5)
        ax = []
        ax.append(fig.add_subplot(gs[0], projection='3d'))
        ax.append(fig.add_subplot(gs[1], projection='3d'))

        p*=1e-24
        p2*=1e-24
        wl = h*1e24
        x = np.linspace(0, 2*wl, npnts)
        c2 = (1-c1**2)**.5
        psi1 = np.exp(1j*p/hbar*x)*c1
        psi2 = np.exp(1j*p2/hbar*x)*c2
        psi = psi1+psi2
        y = np.real(psi)
        z = np.imag(psi)

        x_comb = x
        y_comb = y
        z_comb = z
        c = np.ones_like(x)*-1

        if Real:
            x_comb = np.concatenate([x_comb, x])
            y_comb = np.concatenate([y_comb, y])
            z_comb = np.concatenate([z_comb, np.zeros_like(z)])
            c = np.concatenate([c, np.zeros_like(x)])
                    
        if Imaginary:
            x_comb = np.concatenate([x_comb,x])
            y_comb = np.concatenate([y_comb,  np.zeros_like(y)])
            z_comb = np.concatenate([z_comb, z])
            c = np.concatenate([c, np.ones_like(x)])    

        pos = np.abs(int(npnts *pos/(2*wl*1e9))-1)
        x_pos = np.array([x[pos],x[pos]])
        y_pos = np.array([0., y[pos]])
        z_pos = np.array([0., z[pos]])

        ax[0].plot(x*1e9, np.zeros_like(x), np.zeros_like(x), c='k')
        ax[0].plot(x_pos*1e9, y_pos, z_pos, c='red')
        ax[0].scatter(x_comb*1e9, y_comb, z_comb, s=.1, alpha=1, c=c, vmin=-1, vmax=1,cmap=cmap2)
        ax[0].scatter(x[pos]*1e9,0,0, c='k')
        if Real:
            ax[0].scatter(x[pos]*1e9,y[pos],0, c='tab:blue')
        if Imaginary:
            ax[0].scatter(x[pos]*1e9,0,z[pos], c='tab:orange')
        

        ax[0].view_init(elev=elev, azim=-azim)
        ax[1].view_init(elev=elev, azim=-azim)
        ax[0].set_ylim(-1,1)
        ax[0].set_zlim(-1,1)
        ax[0].set_xlabel('x (nm)')
        ax[0].set_ylabel(f'Real($\\psi$)')
        ax[0].set_zlabel(f'Imag($\\psi$)',labelpad=-.5)

        ax[1].plot(x*1e9, np.zeros_like(x), c='k')
        z = np.abs(psi)**2
        ax[1].plot(x*1e9, z,np.zeros_like(x), label=f'Real($\\psi$)')
        ax[1].scatter(x[pos]*1e9,z[pos],0,c='tab:orange')
        ax[1].set_xlabel('x (nm)')
        ax[1].set_ylabel(f'Real($\\psi$)')
        ax[1].set_zlabel(f'Real($\\psi$)')
        ax[1].set_zlim(-.1, .1)
        ax[1].set_ylim(-.1,3)
        ax[1].set_title(f'$\\psi^*\\psi$')

        wl = h/p*1e9

        #display(Math(rf"\psi = e^{{i p x/\hbar}}= e^{{i 2\pi x/({wl:.2f}nm)}}"))
        ax[0].set_title(f"$\\psi = c_1 e^{{i p_1 x/\\hbar}}+ c_2 e^{{i p_2 x/\\hbar}}$")
        print(rf'c1={c1:.2f}, c2={c2:.2f}')
        plt.show()


In [192]:
def ring_part():
    @interact(
    m = widgets.FloatSlider(min=-15, max=15, step=1, value=3),
    elevation = widgets.FloatSlider(min=0, max=90, value=30),
    rotation = widgets.FloatSlider(min=0, max=180, value=30),
    angle = widgets.FloatSlider(min=0, max=np.pi*2, step=.01, value=0)
    )
    def g(m, elevation, rotation, angle):
        fig = plt.figure()
        gs = GridSpec(1,1,fig)
        ax = fig.add_subplot(gs[0],projection='3d')
        #ax1 = fig.add_subplot(gs[1],projection='3d')
        
        npnts = 1000
        ind = int(npnts*angle/2/np.pi)
        theta = np.linspace(0, 2*np.pi, npnts+1)
        x0 = np.cos(theta)
        y0 = np.sin(theta)
        z0 = np.zeros_like(x0)

        a = .2
        dx = np.sin(theta*m)*a*np.cos(theta)
        dy = np.sin(theta*m)*a*np.sin(theta)
        dz =np.cos(theta*m)*a
        x = x0 +dx
        y =y0+ dy
        z = z0 +dz
        xc = np.concat([x, x0])
        yc = np.concat([y,y0])
        zc = np.concat([z,z0])

        curl = np.ones_like(x)
        ring = np.zeros_like(x0)
        mask = np.concat([curl, ring])

        dx1 = np.sin(-theta*m)*a*np.cos(theta)
        dy1 = np.sin(-theta*m)*a*np.sin(theta)
        dz1 =np.cos(-theta*m)*a
        x1 = x0 +dx1
        y1 =y0+ dy1
        z1 = z0 + dz1
        xc1 = np.concat([x1,x0])
        yc1 = np.concat([y1,y0])
        zc1 = np.concat([z1,z0])

        px = [x0[ind], x[ind]]
        py = [y0[ind], y[ind]]
        pz = [z0[ind], z[ind]]


        p1x = [x0[ind], x1[ind]]
        p1y = [y0[ind], y1[ind]]
        p1z = [z0[ind], z1[ind]]
        ax.scatter(xc, yc, zc, s=.3, c=mask, cmap=cmap)
        ax.plot(px, py, pz, c='tab:blue')
        ax.set_zlim(-1,1)

        ax.view_init(elevation, azim=rotation)

        if np.sin(m*theta[ind]) >= 0:
            string = f'+ i{np.sin(m*theta[ind]):.2f}'
            string1 = f'- i{np.sin(m*theta[ind]):.2f}'
        else:
            string = f'- i{-np.sin(m*theta[ind]):.2f}'
            string1 = f'+ i{-np.sin(m*theta[ind]):.2f}'
        ax.set_title(f"$\\theta$: {angle*180/np.pi:.0f}$^\\circ$\n $e^{{i ({m:.0f}) \\theta}}= {np.cos(m*theta[ind]):.2f} $"+string)
        

        plt.show()
ring_part()

interactive(children=(FloatSlider(value=3.0, description='m', max=15.0, min=-15.0, step=1.0), FloatSlider(valu…

In [193]:
def super_ring_part():
    @interact(
    m = widgets.FloatSlider(min=-5, max=5, step=1, value=3, description='m1'),
    m2 = widgets.FloatSlider(min=-5, max=5, step=1, value=-3,description='m2'),
    c1s= widgets.FloatSlider(min = 0, max=1, value=1, step=.01, description='c1^2'),
    elevation = widgets.FloatSlider(min=0, max=90, value=30),
    rotation = widgets.FloatSlider(min=0, max=180, value=30),
    angle = widgets.FloatSlider(min=0, max=np.pi*2, step=.01, value=0)
    )
    def g(m, m2, c1s, elevation, rotation, angle):
        c1 = c1s**.5
        fig = plt.figure()
        gs = GridSpec(1,2,fig)
        ax = fig.add_subplot(gs[0],projection='3d')
        ax1 = fig.add_subplot(gs[1],projection='3d')
        
        c2 = (1-c1**2)**.5
        
        r =5

        npnts = 1000
        ind = int(npnts*angle/2/np.pi)
        theta = np.linspace(0, 2*np.pi, npnts+1)
        x0 = np.cos(theta)*r
        y0 = np.sin(theta)*r
        z0 = np.zeros_like(x0)*r

        a = 1
        dx = c1*np.sin(theta*m)*a*np.cos(theta)+c2*np.sin(theta*m2)*a*np.cos(theta)
        dy = c1*np.sin(theta*m)*a*np.sin(theta)+c2*np.sin(theta*m2)*a*np.sin(theta)
        dz =c1*np.cos(theta*m)*a+c2*np.cos(theta*m2)*a
        x = x0 +dx
        y =y0+ dy
        z = z0 +dz
        xc = np.concat([x, x0])
        yc = np.concat([y,y0])
        zc = np.concat([z,z0])

        psi = c1*np.exp(1j*m*theta) + c2*np.exp(1j*m2*theta)
        dzs =a**2 * np.abs(psi)**2
        xs = x0 
        ys =y0
        zs = z0 +dzs
        xcs = np.concat([xs, x0])
        ycs = np.concat([ys,y0])
        zcs = np.concat([zs,z0])

        curl = np.ones_like(x)
        ring = np.zeros_like(x0)
        mask = np.concat([curl, ring])

        px = [x0[ind], x[ind]]
        py = [y0[ind], y[ind]]
        pz = [z0[ind], z[ind]]

        ax.scatter(xc, yc, zc, s=.3, c=mask, cmap=cmap)
        ax.plot(px, py, pz, c='tab:blue')
        ax.set_zlim(-r-1,r+1)
        ax.set_xlim(-r-1,r+1)
        ax.set_ylim(-r-1,r+1)

        

        ax.view_init(elevation, azim=rotation)
        ax1.view_init(elevation, azim=rotation)

        ax1.scatter(xcs,ycs, zcs, s=.3, c=mask, cmap=cmap)
        ax1.set_zlim(0, r+1)

        if np.sin(m*theta[ind]) >= 0:
            string = f'+ i{np.sin(m*theta[ind]):.2f}'
            string1 = f'- i{np.sin(m*theta[ind]):.2f}'
        else:
            string = f'- i{-np.sin(m*theta[ind]):.2f}'
            string1 = f'+ i{-np.sin(m*theta[ind]):.2f}'
        #ax.set_title(f"$\\theta$: {angle*180/np.pi:.0f}$^\\circ$\n $e^{{i {m:.0f} \\theta}}= {np.cos(m*theta[ind]):.2f} $"+string)
        ax.set_title(f'$\\psi(\\theta) = c_1 e^{{i ({m}\\theta)}} + c_2 e^{{i ({m2})\\theta}}$')
        print(f'c1={c1:.2f}, c2={c2:.2f}')
        

        plt.show()


## Free particle

In [54]:
free_part()

interactive(children=(FloatSlider(value=0.5, description='p × 10²⁴=', max=3.0, min=-3.0), ToggleButton(value=F…

## Superposition of free particle wavefunctions

In [194]:
super_free_part()

interactive(children=(FloatSlider(value=0.5, description='p1 × 10²⁴=', max=3.0, min=-3.0), FloatSlider(value=-…

## Particle on a ring

In [188]:
ring_part()

interactive(children=(FloatSlider(value=3.0, description='m', max=15.0, min=-15.0, step=1.0), FloatSlider(valu…

## Superposition of particle-on-a-ring wavefunctions

In [195]:
super_ring_part()

interactive(children=(FloatSlider(value=3.0, description='m1', max=5.0, min=-5.0, step=1.0), FloatSlider(value…

## Spherical Coordinates